testes diversos

In [ ]:
%cd /torchfort/examples/fortran/simulation

In [ ]:
%%writefile CMakeLists.txt
# Finds the HDF5 package, essential for reading and writing data in HDF5 format.
# We specify that we need the Fortran component of HDF5.
# The REQUIRED option ensures that the CMake configuration will fail if HDF5 (with Fortran support) is not found.
find_package(HDF5 COMPONENTS Fortran REQUIRED)

# Defines a list of Fortran executable targets that will be built.
# In this case, we have two executables: 'train' and 'train_distributed'.
set(fortran_example_targets
  train
  train_distributed
)

# Adds an executable target named 'train'.
add_executable(train)
# Defines the source code files that will be compiled to create the 'train' executable.
# The PRIVATE keyword indicates that these files are specific to the build of this target
# and should not be exposed to other targets that might link against it.
target_sources(train
  PRIVATE
  train.f90
  simulation.f90
)

# Sets specific properties for the 'train' target.
# In this case, we are setting the directory where the Fortran modules generated during compilation
# should be placed. ${CMAKE_CURRENT_SOURCE_DIR} represents the current directory of the CMakeLists.txt file.
# 'mod/0' is the subdirectory within the source directory where the modules will be stored.
set_target_properties(train PROPERTIES Fortran_MODULE_DIRECTORY ${CMAKE_CURRENT_SOURCE_DIR}/mod/0 )

# Adds another executable target named 'train_distributed'.
add_executable(train_distributed)

# Defines the source code files for the 'train_distributed' executable.
target_sources(train_distributed
  PRIVATE
  train_distributed.f90
  simulation.f90
)

# Sets the properties for the 'train_distributed' target, similar to 'train',
# but with a different Fortran module directory ('mod/1'). This can be useful to avoid
# module name conflicts if the two executables have dependencies on modules with the same name.
set_target_properties(train_distributed PROPERTIES Fortran_MODULE_DIRECTORY ${CMAKE_CURRENT_SOURCE_DIR}/mod/1 )

# Starts a loop over each target listed in 'fortran_example_targets' ('train' and 'train_distributed').
foreach(tgt ${fortran_example_targets})

# Defines the include directories for the current target ($tgt).
# PRIVATE means these directories are only needed during the compilation of this target.
# ${CMAKE_BINARY_DIR}/include: Include directory within the CMake build directory.
#                              May contain header files generated during the build process.
# ${MPI_Fortran_INCLUDE_DIRS}: Include directories required to compile Fortran code that uses MPI (Message Passing Interface).
#                               This variable is usually set by the FindMPI.cmake package.
# ${HDF5_Fortran_INCLUDE_DIRS}: Include directories required to compile Fortran code that uses the HDF5 library.
#                                This variable is set by the find_package(HDF5) we found earlier.
  target_include_directories(${tgt}
    PRIVATE
    ${CMAKE_BINARY_DIR}/include
    ${MPI_Fortran_INCLUDE_DIRS}
    ${HDF5_Fortran_INCLUDE_DIRS}
  )

# Specifies the libraries that the current target ($tgt) should link against.
# PRIVATE means these libraries are only needed for this target.
# MPI::MPI_Fortran: Fortran interface of the MPI library (provided by the FindMPI package).
# hdf5::hdf5_fortran: Fortran interface of the HDF5 library (provided by find_package(HDF5)).
# "${PROJECT_NAME}_fort": An internal Fortran library of the project (name is derived from the project name).
# ${PROJECT_NAME}: An internal C/C++ library of the project (name is the project name defined in the 'project(...)' command).
  target_link_libraries(${tgt} PRIVATE MPI::MPI_Fortran)
  target_link_libraries(${tgt} PRIVATE hdf5::hdf5_fortran)
  target_link_libraries(${tgt} PRIVATE "${PROJECT_NAME}_fort")
  target_link_libraries(${tgt} PRIVATE ${PROJECT_NAME})

# Checks which Fortran compiler is being used.
  if (CMAKE_Fortran_COMPILER_ID STREQUAL "NVHPC")

# Defines specific compilation options for the NVHPC (NVIDIA HPC SDK) compiler.
# $<$<COMPILE_LANGUAGE:Fortran>:...> applies options only when the compile language is Fortran.
# -cpp: Enables the C preprocessor for Fortran files.
# -acc: Enables OpenACC directives for parallel programming on GPUs.
# -gpu=${CUF_GPU_ARG}: Passes a specific GPU argument (the CUF_GPU_ARG variable must be defined elsewhere).
    target_compile_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>:-cpp -acc -gpu=${CUF_GPU_ARG}>)

# Defines specific linking options for the NVHPC compiler.
# The options are similar to the compile options, indicating that linking should also consider GPU acceleration.
    target_link_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>: -acc -gpu=${CUF_GPU_ARG}>)

# Otherwise, if the Fortran compiler is GNU gfortran.
  elseif (CMAKE_Fortran_COMPILER_ID STREQUAL "GNU")

# Defines specific compilation options for the GNU Fortran compiler.
# -cpp: Enables the C preprocessor for Fortran files.
# -fbackslash: Allows the use of backslashes to continue Fortran code lines (common in older code).
# -fopenacc: Enables OpenACC support for parallel programming.
    target_compile_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>:-cpp -fbackslash -fopenacc>)

# Defines specific linking options for the GNU Fortran compiler.
# -fopenacc: Ensures that OpenACC libraries are linked.
    target_link_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>: -fopenacc>)
  endif()

# End of the foreach loop.
endforeach()

# Defines the installation rules for the Fortran executable targets.
install(
  TARGETS ${fortran_example_targets}

# Specifies the destination directory for the executables during installation.
# ${CMAKE_INSTALL_PREFIX} is a user-configurable installation prefix (usually /usr/local or /opt/...).
# The executables will be placed in ${CMAKE_INSTALL_PREFIX}/bin/examples/fortran/simulation.
  RUNTIME DESTINATION ${CMAKE_INSTALL_PREFIX}/bin/examples/fortran/simulation
)

# Defines the installation rules for data files and Python scripts.
install(
  FILES ${CMAKE_CURRENT_SOURCE_DIR}/config_mlp_native.yaml
        ${CMAKE_CURRENT_SOURCE_DIR}/config_fcn_torchscript.yaml
        ${CMAKE_CURRENT_SOURCE_DIR}/generate_fcn_model.py
        ${CMAKE_CURRENT_SOURCE_DIR}/visualize.py

# Specifies the destination directory for these files during installation,
# in the same location as the Fortran executables.
  DESTINATION ${CMAKE_INSTALL_PREFIX}/bin/examples/fortran/simulation)

In [ ]:
%cd /torchfort

In [ ]:
%%writefile CMakeLists.txt
# Define a versão mínima do CMake necessária. A versão 3.17 é especificada no CMakeLists.txt pai
# para suporte ao FindCUDAToolkit, então mantemos essa.
cmake_minimum_required(VERSION 3.17)

# Define o padrão C++ para a compilação. Embora os exemplos sejam Fortran, a biblioteca C++
# que eles linkam utiliza este padrão.
set(CMAKE_CXX_STANDARD 17)

# Define o tipo de build padrão se não for especificado pelo usuário.
if (NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE RelWithDebInfo)
endif()

# Aplica uma política do CMake. Mantida do CMakeLists.txt pai.
cmake_policy(SET CMP0057 NEW)

# --- Opções de Build Definidas pelo Usuário (do CMakeLists.txt pai) ---
# Mantemos apenas as opções relevantes para os exemplos Fortran e suas dependências.
# TORCHFORT_CUDA_CC_LIST: Lista de capacidades de computação CUDA. Usada para gerar CUF_GPU_ARG.
set(TORCHFORT_CUDA_CC_LIST "70;80;90" CACHE STRING "List of CUDA compute capabilities for GPU compilation.")
# TORCHFORT_NCCL_ROOT: Caminho para encontrar a instalação do NCCL. Necessário se usar MPI distribuído com GPU.
set(TORCHFORT_NCCL_ROOT CACHE STRING "Path to search for NCCL installation.")
# TORCHFORT_YAML_CPP_ROOT: Caminho para encontrar a instalação do yaml-cpp. A lib C++ linka contra ela.
set(TORCHFORT_YAML_CPP_ROOT CACHE STRING "Path to search for yaml-cpp installation.")
# TORCHFORT_ENABLE_GPU: Habilita o suporte a GPU/CUDA. Afeta a busca por CUDA/NCCL e opções de compilação.
option(TORCHFORT_ENABLE_GPU "Enable GPU/CUDA support" ON)
# Opções como BUILD_FORTRAN, BUILD_EXAMPLES, BUILD_TESTS são removidas pois este arquivo JÁ constrói os exemplos Fortran.

# Verifica se TORCHFORT_YAML_CPP_ROOT foi definido, caso contrário, mostra erro fatal (mantido do pai).
if (NOT TORCHFORT_YAML_CPP_ROOT)
  message(FATAL_ERROR "Please set TORCHFORT_YAML_CPP_ROOT to yaml-cpp installation directory.")
endif()

# Define os idiomas do projeto. Precisamos de Fortran para os exemplos e CXX para a biblioteca que eles linkam.
project(CombinedFortranExamples LANGUAGES Fortran CXX) # Nome do projeto alterado para refletir o conteúdo

# --- Busca por Dependências Externas (do CMakeLists.txt pai e do exemplo Fortran) ---

# MPI: Necessário para exemplos distribuídos.
find_package(MPI REQUIRED)

# HDF5: Necessário para leitura/escrita de dados nos exemplos.
find_package(HDF5 COMPONENTS Fortran REQUIRED)

# CUDA e NCCL: Necessário se o suporte a GPU estiver habilitado.
if (TORCHFORT_ENABLE_GPU)
  find_package(CUDAToolkit REQUIRED)

  # Busca pelo compilador NVHPC e configurações CMake (do pai).
  find_program(NVHPC_CXX_BIN "nvc++")
  if (NVHPC_CXX_BIN)
    string(REPLACE "compilers/bin/nvc++" "cmake" NVHPC_CMAKE_DIR ${NVHPC_CXX_BIN})
    # Adiciona o diretório CMake do NVHPC ao CMAKE_PREFIX_PATH para ajudar a encontrar outros pacotes NVHPC.
    set(CMAKE_PREFIX_PATH "${CMAKE_PREFIX_PATH};${NVHPC_CMAKE_DIR}")
    find_package(NVHPC COMPONENTS "") # Encontra o pacote NVHPC principal.
  endif()

  # Busca pela biblioteca NCCL (do pai). Usa TORCHFORT_NCCL_ROOT ou as configurações do NVHPC.
  if (TORCHFORT_NCCL_ROOT)
    find_path(NCCL_INCLUDE_DIR REQUIRED
      NAMES nccl.h
      HINTS ${TORCHFORT_NCCL_ROOT}/include
    )
    find_library(NCCL_LIBRARY REQUIRED
      NAMES nccl
      HINTS ${TORCHFORT_NCCL_ROOT}/lib
    )
  else()
    if (NVHPC_FOUND)
      find_package(NVHPC REQUIRED COMPONENTS NCCL) # Tenta encontrar NCCL via NVHPC.
      find_library(NCCL_LIBRARY
        NAMES nccl
        HINTS ${NVHPC_NCCL_LIBRARY_DIR} # Diretório do NCCL fornecido pelo NVHPC.
      )
      # Deriva o diretório de include do diretório da biblioteca NCCL do NVHPC.
      string(REPLACE "/lib" "/include" NCCL_INCLUDE_DIR ${NVHPC_NCCL_LIBRARY_DIR})
    else()
      message(FATAL_ERROR "Cannot find NCCL library. Please set TORCHFORT_NCCL_ROOT or ensure NVHPC is configured.")
    endif()
  endif()
  message(STATUS "Using NCCL library: ${NCCL_LIBRARY}")
endif()

# PyTorch: A biblioteca interna linka contra PyTorch/LibTorch.
find_package(Torch REQUIRED)

# yaml-cpp: A biblioteca interna linka contra yaml-cpp.
# Usa TORCHFORT_YAML_CPP_ROOT para encontrar.
find_path(YAML_CPP_INCLUDE_DIR REQUIRED
  NAMES yaml-cpp/yaml.h
  HINTS ${TORCHFORT_YAML_CPP_ROOT}/include
)
find_library(YAML_CPP_LIBRARY REQUIRED
  NAMES yaml-cpp
  HINTS ${TORCHFORT_YAML_CPP_ROOT}/lib
)
message(STATUS "Using yaml-cpp library: ${YAML_CPP_LIBRARY}")

# --- Definição da Biblioteca Interna C++ (do CMakeLists.txt pai) ---
# Os exemplos Fortran linkam contra esta biblioteca. Incluímos todos os fontes C++
# listados no pai para garantir que a biblioteca seja construída corretamente.
add_library(torchfort SHARED) # Mantém o nome do target original que os exemplos esperam.
set_target_properties(torchfort PROPERTIES LIBRARY_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR}/lib)

# Lista de arquivos fonte C++ (copie os arquivos .cpp necessários para src/csrc/ relativos a este CMakeLists.txt)
target_sources(torchfort
  PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/distributed.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/logging.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/model_state.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/model_wrapper.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/model_pack.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/param_map.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/setup.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/torchfort.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/training.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/utils.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/losses/l1_loss.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/losses/mse_loss.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/losses/torchscript_loss.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/lr_schedulers/cosine_annealing_lr.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/lr_schedulers/multistep_lr.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/lr_schedulers/polynomial_lr.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/lr_schedulers/scheduler_setup.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/lr_schedulers/step_lr.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/lr_schedulers/linear_lr.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/models/mlp_model.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/models/sac_model.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/models/actor_critic_model.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/policy.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/utils.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/off_policy/interface.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/off_policy/ddpg.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/off_policy/td3.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/off_policy/sac.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/on_policy/interface.cpp
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/rl/on_policy/ppo.cpp
)

# Diretórios de include para a biblioteca C++ (do pai).
target_include_directories(torchfort
  PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/include # Inclui os headers da própria lib
  ${YAML_CPP_INCLUDE_DIR} # Inclui headers do yaml-cpp
  ${MPI_CXX_INCLUDE_DIRS} # Inclui headers do MPI C++
  ${TORCH_INCLUDE_DIRS} # Inclui headers do PyTorch
)
if (TORCHFORT_ENABLE_GPU)
  target_include_directories(torchfort
    PRIVATE
    ${CUDAToolkit_INCLUDE_DIRS} # Inclui headers do CUDA Toolkit
    ${NCCL_INCLUDE_DIR} # Inclui headers do NCCL
  )
  target_link_libraries(torchfort PRIVATE CUDA::cudart) # Linka contra runtime CUDA
  target_compile_definitions(torchfort PRIVATE ENABLE_GPU) # Define macro para compilação GPU
endif()

# Linka as bibliotecas necessárias para a biblioteca C++ (do pai).
target_link_libraries(torchfort PRIVATE ${TORCH_LIBRARIES})
target_link_libraries(torchfort PRIVATE ${NCCL_LIBRARY})
target_link_libraries(torchfort PRIVATE MPI::MPI_CXX) # Linka contra interface C++ do MPI
target_link_libraries(torchfort PRIVATE ${YAML_CPP_LIBRARY})

# Opções de compilação para a biblioteca C++ (do pai).
target_compile_definitions(torchfort PRIVATE YAML_CPP_STATIC_DEFINE)
target_compile_options(torchfort PRIVATE $<$<COMPILE_LANGUAGE:CXX>:${TORCH_CXX_FLAGS}>)

# Headers públicos da biblioteca C++ (copie para src/csrc/include/ relativos a este CMakeLists.txt)
set(public_headers
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/include/torchfort.h
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/include/torchfort_rl.h
  ${CMAKE_CURRENT_SOURCE_DIR}/src/csrc/include/torchfort_enums.h
)
set_target_properties("torchfort" PROPERTIES PUBLIC_HEADER "${public_headers}")

# Regras de instalação para a biblioteca C++ (se desejar instalar)
install(
  TARGETS torchfort
  EXPORT "torchfortTargets"
  PUBLIC_HEADER DESTINATION ${CMAKE_INSTALL_PREFIX}/include
  INCLUDES DESTINATION ${CMAKE_INSTALL_PREFIX}/include # Instala headers em include
  RUNTIME DESTINATION ${CMAKE_INSTALL_PREFIX}/lib # Instala a biblioteca compartilhada em lib
)


# --- Definição da Biblioteca/Módulo Fortran (do CMakeLists.txt pai) ---
# Os exemplos Fortran também linkam contra este módulo/biblioteca Fortran.
add_library(torchfort_fort SHARED) # Mantém o nome do target original.
set_target_properties(torchfort_fort PROPERTIES LIBRARY_OUTPUT_DIRECTORY ${CMAKE_BINARY_DIR}/lib)
# Define onde o compilador Fortran deve colocar os arquivos de módulo (.mod).
set_target_properties(torchfort_fort PROPERTIES Fortran_MODULE_DIRECTORY ${CMAKE_BINARY_DIR}/include) # Coloca .mod em CMAKE_BINARY_DIR/include

# Opções de compilação Fortran para a biblioteca (do pai).
if (CMAKE_Fortran_COMPILER_ID STREQUAL "NVHPC")
  # Cria a string de argumento -gpu para compilação GPU com nvfortran (do pai).
  # Depende de TORCHFORT_CUDA_CC_LIST e NVHPC_CUDA_VERSION.
  set(CUF_GPU_ARG "") # Inicializa a variável antes de usar APPEND
  foreach(CUDA_CC ${TORCHFORT_CUDA_CC_LIST})
    list(APPEND CUF_GPU_ARG "cc${CUDA_CC}")
  endforeach()
  if (NVHPC_CUDA_VERSION) # Verifica se a variável existe antes de usar
    list(APPEND CUF_GPU_ARG "cuda${NVHPC_CUDA_VERSION}")
  endif()
  list(JOIN CUF_GPU_ARG "," CUF_GPU_ARG)
  message(STATUS "Generated CUF_GPU_ARG: ${CUF_GPU_ARG}") # Mostra o argumento gerado

  target_compile_options(torchfort_fort PRIVATE $<$<COMPILE_LANGUAGE:Fortran>:-cpp -cuda -gpu=${CUF_GPU_ARG}>)
elseif (CMAKE_Fortran_COMPILER_ID STREQUAL "GNU")
  target_compile_options(torchfort_fort PRIVATE $<$<COMPILE_LANGUAGE:Fortran>:-cpp>)
endif()

# Testa se MPI_Comm_f2c/c2f está disponível (do pai). Requer o arquivo test_mpi_f2c.f90.
try_compile(
  TEST_F2C_RESULT
  ${CMAKE_BINARY_DIR}
  ${CMAKE_CURRENT_SOURCE_DIR}/cmake/test_mpi_f2c.f90 # Caminho relativo ao novo CMakeLists.txt
  LINK_LIBRARIES MPI::MPI_Fortran
)
if (NOT TEST_F2C_RESULT)
  message(STATUS "Could not link MPI_Comm_f2c in Fortran module. Setting -DMPICH flag during module compilation.")
  target_compile_definitions(torchfort_fort PRIVATE MPICH)
endif()

# Arquivos fonte Fortran para a biblioteca (copie torchfort_m.F90 para src/fsrc/ relativo)
target_sources(torchfort_fort
  PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/src/fsrc/torchfort_m.F90
)
# Linka a biblioteca Fortran contra MPI Fortran (do pai).
target_link_libraries(torchfort_fort PRIVATE MPI::MPI_Fortran)

# Regras de instalação para a biblioteca Fortran e módulo (se desejar instalar)
install(
  TARGETS torchfort_fort
  RUNTIME DESTINATION ${CMAKE_INSTALL_PREFIX}/lib # Instala a biblioteca compartilhada em lib
)
# Instala o arquivo de módulo Fortran (.mod) no diretório de includes.
install(FILES ${CMAKE_BINARY_DIR}/include/torchfort.mod DESTINATION ${CMAKE_INSTALL_PREFIX}/include)


# --- Definição dos Executáveis de Exemplo Fortran (do primeiro CMakeLists.txt) ---

# Adiciona o alvo executável 'train'.
add_executable(train)
# Define os arquivos de código fonte (copie para examples/fortran/graph/ relativo)
target_sources(train
  PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/graph/train.f90
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/graph/simulation.f90 # Supondo que esta é a cópia usada
)
# Define o diretório para os módulos Fortran gerados por este alvo.
set_target_properties(train PROPERTIES Fortran_MODULE_DIRECTORY ${CMAKE_BINARY_DIR}/mod/train) # Coloca módulos em um dir de build específico


# Adiciona o alvo executável 'train_distributed'.
add_executable(train_distributed)
# Define os arquivos de código fonte (copie para examples/fortran/simulation/ relativo)
target_sources(train_distributed
  PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/simulation/train_distributed.f90
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/simulation/simulation.f90 # Supondo que esta é a cópia usada
)
# Define o diretório para os módulos Fortran gerados por este alvo.
set_target_properties(train_distributed PROPERTIES Fortran_MODULE_DIRECTORY ${CMAKE_BINARY_DIR}/mod/train_distributed) # Coloca módulos em outro dir de build específico

# Lista de alvos executáveis para facilitar iteração ou instalação.
set(fortran_example_targets train train_distributed)


# --- Configurações Comuns para os Executáveis (do loop foreach no primeiro CMakeLists.txt) ---
foreach(tgt ${fortran_example_targets})

  # Define os diretórios de inclusão para o alvo atual.
  # Inclui onde encontrar módulos Fortran gerados (o nosso BINAry_DIR/include),
  # headers do MPI Fortran e headers do HDF5 Fortran.
  target_include_directories(${tgt}
    PRIVATE
    ${CMAKE_BINARY_DIR}/include # Para encontrar torchfort.mod e outros módulos gerados
    ${MPI_Fortran_INCLUDE_DIRS} # Headers MPI para Fortran
    ${HDF5_Fortran_INCLUDE_DIRS} # Headers HDF5 para Fortran
  )

  # Especifica as bibliotecas com as quais o alvo atual deve ser vinculado.
  # Linka contra MPI Fortran, HDF5 Fortran, a biblioteca interna Fortran e a biblioteca interna C++.
  target_link_libraries(${tgt} PRIVATE MPI::MPI_Fortran)
  target_link_libraries(${tgt} PRIVATE hdf5::hdf5_fortran)
  target_link_libraries(${tgt} PRIVATE torchfort_fort) # Linka contra a lib Fortran que definimos acima
  target_link_libraries(${tgt} PRIVATE torchfort) # Linka contra a lib C++ que definimos acima

  # Opções de compilação/linkagem específicas do compilador (do primeiro CMakeLists.txt).
  if (CMAKE_Fortran_COMPILER_ID STREQUAL "NVHPC")
    # Usa o CUF_GPU_ARG gerado anteriormente.
    target_compile_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>:-cpp -acc -gpu=${CUF_GPU_ARG}>)
    target_link_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>: -acc -gpu=${CUF_GPU_ARG}>)
  elseif (CMAKE_Fortran_COMPILER_ID STREQUAL "GNU")
    target_compile_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>:-cpp -fbackslash -fopenacc>)
    target_link_options(${tgt} PRIVATE $<$<COMPILE_LANGUAGE:Fortran>: -fopenacc>)
  endif()

endforeach()


# --- Regras de Instalação (do primeiro CMakeLists.txt) ---

# Instala os executáveis Fortran.
install(
  TARGETS ${fortran_example_targets}
  # Define o diretório de destino relativo ao prefixo de instalação.
  RUNTIME DESTINATION ${CMAKE_INSTALL_PREFIX}/bin/examples/fortran/simulation
)

# Instala arquivos de dados e scripts Python necessários para os exemplos.
# (copie para examples/fortran/simulation/ relativos a este CMakeLists.txt)
install(
  FILES
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/simulation/config_mlp_native.yaml
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/simulation/config_fcn_torchscript.yaml
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/simulation/generate_fcn_model.py
  ${CMAKE_CURRENT_SOURCE_DIR}/examples/fortran/simulation/visualize.py
  # Define o diretório de destino, o mesmo dos executáveis.
  DESTINATION ${CMAKE_INSTALL_PREFIX}/bin/examples/fortran/simulation
)

In [ ]:
%%bash
mkdir build && cd build
cmake .. \
  -DMPI_ROOT=/caminho/para/sua/instalacao/mpi \
  -DHDF5_ROOT=/caminho/para/sua/instalacao/hdf5 \
  -DTORCH_DIR=/caminho/para/sua/instalacao/libtorch/share/cmake/Torch \
  -DTORCHFORT_YAML_CPP_ROOT=/caminho/para/sua/instalacao/yaml-cpp \
  -DTORCHFORT_NCCL_ROOT=/caminho/para/sua/instalacao/nccl \
  -DTORCHFORT_CUDA_CC_LIST="70;80;96" \ # Exemplo, ajuste conforme sua GPU
  -DCMAKE_INSTALL_PREFIX=/caminho/para/onde/instalar
make